# 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

#Plotting style
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d2e',
    'axes.edgecolor':   '#2e3250',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'figure.titlesize': 16,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
    'legend.facecolor': '#1a1d2e',
    'legend.edgecolor': '#2e3250',
})

SENTIMENT_COLORS = {
    'Extreme Fear':  '#ff4d4d',
    'Fear':          '#ff9966',
    'Neutral':       '#aaaacc',
    'Greed':         '#66cc88',
    'Extreme Greed': '#00e676',
}
SENTIMENT_ORDER = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']

print('Setup complete')

## 2. Load & Preview Datasets

In [ ]:
#Load Fear & Greed Index
fg = pd.read_csv('data/fear_greed_index.csv')
fg['date'] = pd.to_datetime(fg['date'])
fg = fg.sort_values('date').reset_index(drop=True)

print(f"Fear & Greed Index: {fg.shape[0]:,} days | {fg['date'].min().date()} → {fg['date'].max().date()}")
fg.head(3)

In [ ]:
#Load Trader Data
tr = pd.read_csv('data/historical_data.csv')
tr['date'] = pd.to_datetime(tr['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce').dt.normalize()
tr = tr.dropna(subset=['date'])

print(f"Trader Data: {tr.shape[0]:,} rows | {tr['date'].min().date()} → {tr['date'].max().date()}")
print(f"Unique Accounts: {tr['Account'].nunique():,}")
print(f"Unique Coins: {tr['Coin'].nunique():,}")
tr.head(3)

## 3.  Exploratory Data Analysis

### 3.1 Fear & Greed Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Bitcoin Fear & Greed Index — Overview', fontsize=15, fontweight='bold', y=1.02)

#Distribution bar chart
counts = fg['classification'].value_counts().reindex(SENTIMENT_ORDER)
colors = [SENTIMENT_COLORS[s] for s in SENTIMENT_ORDER]
axes[0].bar(SENTIMENT_ORDER, counts.values, color=colors, edgecolor='#0f1117', linewidth=0.8)
axes[0].set_title('Sentiment Day Distribution')
axes[0].set_ylabel('Number of Days')
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=9, color='white')

#Time series of index value
axes[1].plot(fg['date'], fg['value'], color='#58a6ff', linewidth=0.6, alpha=0.8)
axes[1].axhline(25, color='#ff4d4d', linestyle='--', linewidth=0.8, label='Extreme Fear (25)')
axes[1].axhline(75, color='#00e676', linestyle='--', linewidth=0.8, label='Extreme Greed (75)')
axes[1].fill_between(fg['date'], fg['value'], 50, where=(fg['value'] < 50),
                      color='#ff4d4d', alpha=0.1)
axes[1].fill_between(fg['date'], fg['value'], 50, where=(fg['value'] >= 50),
                      color='#00e676', alpha=0.1)
axes[1].set_title('Fear & Greed Index Over Time')
axes[1].set_ylabel('Index Value (0–100)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('plots/01_fear_greed_overview.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Plot saved')

### 3.2 Trader Data EDA

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Trader Data — Exploratory Analysis', fontsize=15, fontweight='bold')

#Top 10 coins by trade count
top_coins = tr['Coin'].value_counts().head(10)
axes[0,0].barh(top_coins.index[::-1], top_coins.values[::-1], color='#58a6ff', edgecolor='#0f1117')
axes[0,0].set_title('Top 10 Most Traded Coins')
axes[0,0].set_xlabel('Number of Trades')

#PnL distribution (clipped)
pnl_closed = tr[tr['Closed PnL'] != 0]['Closed PnL']
pnl_clipped = pnl_closed.clip(-500, 500)
axes[0,1].hist(pnl_clipped, bins=80, color='#58a6ff', edgecolor='#0f1117', alpha=0.85)
axes[0,1].axvline(0, color='#ff4d4d', linewidth=1.5, linestyle='--', label='Break-even')
axes[0,1].set_title('Closed PnL Distribution (clipped ±$500)')
axes[0,1].set_xlabel('Closed PnL ($)')
axes[0,1].set_ylabel('Frequency')
axes[0,1].legend()

#Buy vs Sell
side_counts = tr['Side'].value_counts()
axes[1,0].pie(side_counts.values, labels=side_counts.index,
              colors=['#58a6ff','#ff9966'], autopct='%1.1f%%',
              startangle=90, textprops={'color':'white'})
axes[1,0].set_title('Buy vs Sell Trade Distribution')

#Trade size USD distribution
size_clipped = tr['Size USD'].clip(0, 50000)
axes[1,1].hist(size_clipped, bins=60, color='#66cc88', edgecolor='#0f1117', alpha=0.85)
axes[1,1].set_title('Trade Size Distribution (clipped $50K)')
axes[1,1].set_xlabel('Size USD ($)')
axes[1,1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('plots/02_trader_eda.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 4. Merging Datasets on Date

In [ ]:
#Merge on date
merged = tr.merge(fg[['date', 'value', 'classification']], on='date', how='inner')
merged['is_win']         = merged['Closed PnL'] > 0
merged['is_closed']      = merged['Closed PnL'] != 0
merged['sentiment_order'] = merged['classification'].map({s: i for i, s in enumerate(SENTIMENT_ORDER)})

print(f"Merged dataset: {merged.shape[0]:,} trades across {merged['date'].nunique():,} days")
print()
print("Trades per Sentiment:")
print(merged['classification'].value_counts().reindex(SENTIMENT_ORDER))

## 5. Sentiment vs Trader Performance

### 5.1 Average PnL by Sentiment

In [ ]:
avg_pnl = merged.groupby('classification')['Closed PnL'].mean().reindex(SENTIMENT_ORDER)
std_pnl = merged.groupby('classification')['Closed PnL'].std().reindex(SENTIMENT_ORDER)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [SENTIMENT_COLORS[s] for s in SENTIMENT_ORDER]
bars = ax.bar(SENTIMENT_ORDER, avg_pnl.values, color=colors,
              yerr=std_pnl.values/np.sqrt(merged['classification'].value_counts().reindex(SENTIMENT_ORDER)),
              capsize=5, edgecolor='#0f1117', linewidth=0.8, error_kw={'color':'white','alpha':0.5})

ax.axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_title('Average Closed PnL by Market Sentiment', fontsize=14, fontweight='bold')
ax.set_ylabel('Average Closed PnL ($)')
ax.set_xlabel('Market Sentiment')
for bar, val in zip(bars, avg_pnl.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'${val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/03_avg_pnl_by_sentiment.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

### 5.2 Win Rate by Sentiment

In [ ]:
closed_trades = merged[merged['is_closed']]
win_rate = closed_trades.groupby('classification')['is_win'].mean().reindex(SENTIMENT_ORDER) * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(SENTIMENT_ORDER, win_rate.values, color=colors, edgecolor='#0f1117', linewidth=0.8)
ax.axhline(50, color='white', linewidth=1, linestyle='--', alpha=0.6, label='50% baseline')
ax.set_title('Win Rate (%) by Market Sentiment', fontsize=14, fontweight='bold')
ax.set_ylabel('Win Rate (%)')
ax.set_ylim(0, 100)
ax.legend()
for bar, val in zip(bars, win_rate.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/04_win_rate_by_sentiment.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

### 5.3 Trade Volume & Size by Sentiment

In [ ]:
avg_size = merged.groupby('classification')['Size USD'].mean().reindex(SENTIMENT_ORDER)
trade_count = merged.groupby('classification').size().reindex(SENTIMENT_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Trading Activity by Market Sentiment', fontsize=14, fontweight='bold')

axes[0].bar(SENTIMENT_ORDER, avg_size.values, color=colors, edgecolor='#0f1117', linewidth=0.8)
axes[0].set_title('Average Trade Size (USD)')
axes[0].set_ylabel('Average Size ($)')
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(avg_size.values):
    axes[0].text(i, v + 50, f'${v:,.0f}', ha='center', fontsize=9)

axes[1].bar(SENTIMENT_ORDER, trade_count.values, color=colors, edgecolor='#0f1117', linewidth=0.8)
axes[1].set_title('Number of Trades')
axes[1].set_ylabel('Trade Count')
axes[1].tick_params(axis='x', rotation=20)
for i, v in enumerate(trade_count.values):
    axes[1].text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('plots/05_volume_by_sentiment.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 6. Hidden Pattern Discovery

### 6.1 Buy vs Sell Behaviour by Sentiment

In [ ]:
side_sentiment = pd.crosstab(merged['classification'], merged['Side'], normalize='index') * 100
side_sentiment = side_sentiment.reindex(SENTIMENT_ORDER)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(SENTIMENT_ORDER))
width = 0.35
ax.bar(x - width/2, side_sentiment['BUY'], width, label='BUY', color='#58a6ff', edgecolor='#0f1117')
ax.bar(x + width/2, side_sentiment['SELL'], width, label='SELL', color='#ff9966', edgecolor='#0f1117')
ax.set_xticks(x)
ax.set_xticklabels(SENTIMENT_ORDER, rotation=15)
ax.set_title('Buy vs Sell Ratio by Market Sentiment', fontsize=14, fontweight='bold')
ax.set_ylabel('Percentage of Trades (%)')
ax.axhline(50, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
ax.legend()
plt.tight_layout()
plt.savefig('plots/06_buy_sell_by_sentiment.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print(side_sentiment.round(1))

### 6.2 Top Coin Performance by Sentiment

In [ ]:
top_coins_list = tr['Coin'].value_counts().head(6).index.tolist()
coin_sentiment_pnl = (merged[merged['Coin'].isin(top_coins_list)]
                      .groupby(['Coin', 'classification'])['Closed PnL']
                      .mean()
                      .unstack()
                      .reindex(columns=SENTIMENT_ORDER))

fig, ax = plt.subplots(figsize=(12, 6))
coin_sentiment_pnl.plot(kind='bar', ax=ax,
                         color=[SENTIMENT_COLORS[s] for s in SENTIMENT_ORDER],
                         edgecolor='#0f1117', linewidth=0.5)
ax.set_title('Average PnL by Coin & Sentiment (Top 6 Coins)', fontsize=14, fontweight='bold')
ax.set_ylabel('Average Closed PnL ($)')
ax.set_xlabel('Coin')
ax.tick_params(axis='x', rotation=15)
ax.legend(title='Sentiment', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('plots/07_coin_sentiment_pnl.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

### 6.3 Contrarian Traders — Who Profits in Fear Markets?

In [ ]:
#Traders who have > 20 closed trades in Fear sentiment
fear_trades = merged[(merged['classification'].isin(['Fear', 'Extreme Fear'])) & (merged['is_closed'])]
fear_perf = fear_trades.groupby('Account').agg(
    fear_pnl   = ('Closed PnL', 'sum'),
    fear_trades= ('Closed PnL', 'count'),
    fear_winrate=('is_win', 'mean')
).query('fear_trades >= 20').sort_values('fear_pnl', ascending=False)

#Overall performance
all_perf = merged[merged['is_closed']].groupby('Account').agg(
    total_pnl   = ('Closed PnL', 'sum'),
    total_trades= ('Closed PnL', 'count'),
    overall_wr  = ('is_win', 'mean')
)

contrarian = fear_perf.head(10).join(all_perf, how='left')
contrarian['fear_pnl_pct'] = (contrarian['fear_pnl'] / contrarian['total_pnl'] * 100).round(1)
contrarian.index = [f"{a[:6]}...{a[-4:]}" for a in contrarian.index]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(contrarian.index[::-1], contrarian['fear_pnl'][::-1],
               color='#ff9966', edgecolor='#0f1117', linewidth=0.8)
ax.set_title('Top 10 Contrarian Traders — PnL During Fear Markets', fontsize=13, fontweight='bold')
ax.set_xlabel('Total PnL in Fear Conditions ($)')
for bar, val in zip(bars, contrarian['fear_pnl'][::-1]):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('plots/08_contrarian_traders.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print(contrarian[['fear_pnl','fear_trades','fear_winrate','total_pnl','fear_pnl_pct']].head(10).to_string())

### 6.4 Correlation — Sentiment Value vs Daily PnL

In [ ]:
daily = merged.groupby('date').agg(
    total_pnl   = ('Closed PnL', 'sum'),
    avg_pnl     = ('Closed PnL', 'mean'),
    trade_count = ('Closed PnL', 'count'),
    win_rate    = ('is_win', 'mean'),
    fg_value    = ('value', 'first'),
    sentiment   = ('classification', 'first')
).reset_index()

corr_pnl, p_pnl   = stats.pearsonr(daily['fg_value'], daily['avg_pnl'])
corr_wr,  p_wr    = stats.pearsonr(daily['fg_value'], daily['win_rate'])
corr_vol, p_vol   = stats.pearsonr(daily['fg_value'], daily['trade_count'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Fear & Greed Value vs Daily Trading Metrics', fontsize=14, fontweight='bold')

metrics = [
    ('avg_pnl',    'Daily Avg PnL ($)',     corr_pnl, p_pnl,  '#58a6ff'),
    ('win_rate',   'Daily Win Rate',         corr_wr,  p_wr,   '#66cc88'),
    ('trade_count','Daily Trade Count',      corr_vol, p_vol,  '#ffb347'),
]
for ax, (col, label, corr, pval, color) in zip(axes, metrics):
    sc = ax.scatter(daily['fg_value'], daily[col], c=daily['fg_value'],
                    cmap='RdYlGn', alpha=0.65, s=30, edgecolors='none')
    m, b = np.polyfit(daily['fg_value'], daily[col], 1)
    xline = np.linspace(daily['fg_value'].min(), daily['fg_value'].max(), 100)
    ax.plot(xline, m*xline+b, color=color, linewidth=2)
    sig = '✅ Significant' if pval < 0.05 else '❌ Not Significant'
    ax.set_title(f'{label}\nr={corr:.3f}, p={pval:.3f} | {sig}', fontsize=10)
    ax.set_xlabel('Fear & Greed Index Value')
    ax.set_ylabel(label)
plt.colorbar(sc, ax=axes[-1], label='F&G Value')
plt.tight_layout()
plt.savefig('plots/09_correlation_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

### 6.5 PnL Heatmap — Sentiment × Coin

In [ ]:
top10_coins = tr['Coin'].value_counts().head(10).index
heatmap_data = (merged[merged['Coin'].isin(top10_coins)]
                .groupby(['Coin', 'classification'])['Closed PnL']
                .mean()
                .unstack()
                .reindex(columns=SENTIMENT_ORDER)
                .fillna(0))

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn',
            center=0, linewidths=0.5, linecolor='#0f1117',
            ax=ax, annot_kws={'size': 9})
ax.set_title('Average PnL Heatmap — Top 10 Coins × Sentiment', fontsize=14, fontweight='bold')
ax.set_xlabel('Market Sentiment')
ax.set_ylabel('Coin')
plt.tight_layout()
plt.savefig('plots/10_pnl_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

### 6.6 Daily PnL Timeline with Sentiment Overlay

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
fig.suptitle('Daily Trading PnL vs Fear & Greed Index Over Time', fontsize=14, fontweight='bold')

#Top: FG index
axes[0].plot(daily['date'], daily['fg_value'], color='#58a6ff', linewidth=1)
axes[0].fill_between(daily['date'], daily['fg_value'], 50,
                      where=(daily['fg_value'] < 50), color='#ff4d4d', alpha=0.2)
axes[0].fill_between(daily['date'], daily['fg_value'], 50,
                      where=(daily['fg_value'] >= 50), color='#00e676', alpha=0.2)
axes[0].set_ylabel('Fear & Greed Index')
axes[0].set_ylim(0, 100)
axes[0].axhline(50, color='white', linewidth=0.5, linestyle='--', alpha=0.4)

#Bottom: daily avg PnL
colors_daily = [SENTIMENT_COLORS.get(s, '#aaaacc') for s in daily['sentiment']]
axes[1].bar(daily['date'], daily['avg_pnl'], color=colors_daily, width=0.8, alpha=0.85)
axes[1].axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
axes[1].set_ylabel('Avg Closed PnL ($)')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.savefig('plots/11_timeline_overlay.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 7. Key Insights & Trading Strategy Recommendations

In [ ]:
#Summary statistics table
summary = merged.groupby('classification').agg(
    Trades       = ('Closed PnL', 'count'),
    Avg_PnL      = ('Closed PnL', lambda x: round(x.mean(), 2)),
    Win_Rate_pct = ('is_win', lambda x: round(x.mean()*100, 1)),
    Avg_Size_USD = ('Size USD', lambda x: round(x.mean(), 0)),
    Total_PnL    = ('Closed PnL', lambda x: round(x.sum(), 0)),
).reindex(SENTIMENT_ORDER)

print("=" * 70)
print("   SUMMARY: Trader Performance by Market Sentiment")
print("=" * 70)
print(summary.to_string())
print("=" * 70)